# MAE visual evaluation and error heatmaps

This notebook scans `code/reconstructions/` for saved triplets (`*_orig.png`, `*_masked.png`, `*_recon.png`), computes image-quality metrics (MSE, PSNR, SSIM, LPIPS if available), produces error heatmaps, and aggregates per-epoch statistics.


In [1]:
import os
import re
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

try:
    import torch
    import lpips  # type: ignore
    _lpips_net = lpips.LPIPS(net='alex')
    _lpips_available = True
except Exception:
    _lpips_available = False
    _lpips_net = None

from skimage.metrics import structural_similarity as ssim

RECONS_DIR = Path('/home/ichalalentarek/Desktop/Deep Learning/code/reconstructions')
OUT_DIR = Path('/home/ichalalentarek/Desktop/Deep Learning/code/recon_eval')
OUT_DIR.mkdir(parents=True, exist_ok=True)

triplet_re = re.compile(r"epoch(\d+)_sample(\d+)_orig\.png")

def load_img(p: Path) -> np.ndarray:
    img = Image.open(p).convert('RGB')
    return np.asarray(img)

# Utility metrics

def compute_mse(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    return float(np.mean((a - b) ** 2))

def compute_psnr(a: np.ndarray, b: np.ndarray, data_range: float = 255.0) -> float:
    mse = compute_mse(a, b)
    if mse == 0:
        return float('inf')
    return float(10 * np.log10((data_range ** 2) / mse))

def compute_ssim(a: np.ndarray, b: np.ndarray) -> float:
    # skimage expects HxWxC in [0,255]; set channel_axis for color
    return float(ssim(a, b, channel_axis=2, data_range=255))

def compute_lpips(a: np.ndarray, b: np.ndarray) -> float:
    if not _lpips_available:
        return np.nan
    # to [-1,1]
    t1 = torch.from_numpy(a).permute(2,0,1).unsqueeze(0).float() / 255.0
    t2 = torch.from_numpy(b).permute(2,0,1).unsqueeze(0).float() / 255.0
    t1 = t1 * 2 - 1
    t2 = t2 * 2 - 1
    with torch.no_grad():
        d = _lpips_net(t1, t2)
    return float(d.squeeze().cpu().numpy())

# Scanning triplets

def find_triplets(root: Path) -> List[Dict]:
    records = []
    for p in sorted(root.glob('epoch*_sample*_orig.png')):
        m = triplet_re.match(p.name)
        if not m:
            continue
        epoch, sample = int(m.group(1)), int(m.group(2))
        base = f"epoch{epoch}_sample{sample}"
        orig = p
        recon = root / f"{base}_recon.png"
        masked = root / f"{base}_masked.png"
        if recon.exists():
            records.append({
                'epoch': epoch,
                'sample': sample,
                'orig': orig,
                'recon': recon,
                'masked': masked if masked.exists() else None
            })
    return records

triplets = find_triplets(RECONS_DIR)
print(f"Found {len(triplets)} triplets")


Found 184 triplets


In [2]:
def show_panel(epoch: int, sample: int, orig: np.ndarray, recon: np.ndarray, masked: np.ndarray | None = None, save: Path | None = None):
    err = np.abs(orig.astype(np.float32) - recon.astype(np.float32))
    err_gray = np.mean(err, axis=2)

    fig, axs = plt.subplots(1, 4 if masked is not None else 3, figsize=(12, 3))
    axs[0].imshow(orig)
    axs[0].set_title(f"Orig e{epoch}s{sample}")
    axs[1].imshow(recon)
    axs[1].set_title("Recon")
    if masked is not None:
        axs[2].imshow(masked)
        axs[2].set_title("Masked")
        ax_err = axs[3]
    else:
        ax_err = axs[2]
    im = ax_err.imshow(err_gray, cmap='magma')
    ax_err.set_title("Error heatmap")
    for ax in axs:
        ax.axis('off')
    fig.colorbar(im, ax=ax_err, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if save is not None:
        fig.savefig(save, dpi=200)
    plt.close(fig)

# Render a few random panels
if len(triplets) > 0:
    subset = triplets[:: max(1, len(triplets)//12) ]  # ~12 evenly spaced
    for rec in subset:
        o = load_img(rec['orig'])
        r = load_img(rec['recon'])
        m = load_img(rec['masked']) if rec['masked'] is not None else None
        out_path = OUT_DIR / f"epoch{rec['epoch']}_sample{rec['sample']}_panel.png"
        show_panel(rec['epoch'], rec['sample'], o, r, m, out_path)
    print(f"Saved {len(subset)} panels in {OUT_DIR}")
else:
    print("No triplets found to visualize.")


Saved 13 panels in /home/ichalalentarek/Desktop/Deep Learning/code/recon_eval


In [3]:
# Aggregate metrics per image and per epoch
rows = []
for rec in triplets:
    orig = load_img(rec['orig'])
    recon = load_img(rec['recon'])
    # Ensure same size
    if orig.shape != recon.shape:
        h = min(orig.shape[0], recon.shape[0])
        w = min(orig.shape[1], recon.shape[1])
        orig = orig[:h, :w]
        recon = recon[:h, :w]
    mse_v = compute_mse(orig, recon)
    psnr_v = compute_psnr(orig, recon)
    ssim_v = compute_ssim(orig, recon)
    lpips_v = compute_lpips(orig, recon)
    rows.append({
        'epoch': rec['epoch'],
        'sample': rec['sample'],
        'mse': mse_v,
        'psnr': psnr_v,
        'ssim': ssim_v,
        'lpips': lpips_v,
        'orig_path': str(rec['orig']),
        'recon_path': str(rec['recon'])
    })

metrics_df = pd.DataFrame(rows)
print(metrics_df.describe(include='all'))
metrics_csv = OUT_DIR / 'metrics_by_image.csv'
metrics_df.to_csv(metrics_csv, index=False)
print(f"Metrics saved to {metrics_csv}")

# Per-epoch aggregation
if len(metrics_df) > 0:
    epoch_df = metrics_df.groupby('epoch').agg({'mse':'mean', 'psnr':'mean', 'ssim':'mean', 'lpips':'mean'}).reset_index()
    epoch_csv = OUT_DIR / 'metrics_by_epoch.csv'
    epoch_df.to_csv(epoch_csv, index=False)
    print(f"Per-epoch metrics saved to {epoch_csv}")

    # Plot curves
    plt.figure(figsize=(10,4))
    ax1 = plt.subplot(1,3,1)
    ax1.plot(epoch_df['epoch'], epoch_df['mse'], marker='o')
    ax1.set_title('MSE ↓')
    ax2 = plt.subplot(1,3,2)
    ax2.plot(epoch_df['epoch'], epoch_df['psnr'], marker='o')
    ax2.set_title('PSNR ↑')
    ax3 = plt.subplot(1,3,3)
    ax3.plot(epoch_df['epoch'], epoch_df['ssim'], marker='o')
    ax3.set_title('SSIM ↑')
    for ax in (ax1, ax2, ax3):
        ax.set_xlabel('Epoch')
        ax.grid(True, ls='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'reconstruction_metrics_curves.png', dpi=200)
    plt.close()
else:
    print('No metrics to aggregate.')


             epoch      sample          mse        psnr        ssim  lpips  \
count   184.000000  184.000000   184.000000  184.000000  184.000000    0.0   
unique         NaN         NaN          NaN         NaN         NaN    NaN   
top            NaN         NaN          NaN         NaN         NaN    NaN   
freq           NaN         NaN          NaN         NaN         NaN    NaN   
mean     23.500000    1.500000  3572.192468   13.026518    0.370207    NaN   
std      13.312142    1.121085  1520.057195    1.963008    0.089487    NaN   
min       1.000000    0.000000  1681.190552   10.517002    0.247087    NaN   
25%      12.000000    0.750000  2347.253418   11.369766    0.291215    NaN   
50%      23.500000    1.500000  3479.248779   12.801378    0.347321    NaN   
75%      35.000000    2.250000  4750.744873   14.425274    0.432096    NaN   
max      46.000000    3.000000  5772.715332   15.874634    0.560865    NaN   

                                                orig_path  \
co

In [4]:
# Focus on a specific epoch (e.g., 46)
TARGET_EPOCH = 46
focus = [rec for rec in triplets if rec['epoch'] == TARGET_EPOCH]
print(f"Found {len(focus)} samples for epoch {TARGET_EPOCH}")

for rec in focus:
    o = load_img(rec['orig'])
    r = load_img(rec['recon'])
    m = load_img(rec['masked']) if rec['masked'] is not None else None
    save = OUT_DIR / f"epoch{TARGET_EPOCH}_sample{rec['sample']}_panel.png"
    show_panel(rec['epoch'], rec['sample'], o, r, m, save)
print("Panels for target epoch saved.")


Found 4 samples for epoch 46
Panels for target epoch saved.
